In [92]:
import gymnasium as gym
import numpy as np
import torch

from collections import deque
import torch.optim as optim
from torch.distributions import Categorical

In [96]:
import torch.nn.functional as F
probs = F.softmax(torch.randn(1, 2))
print(probs)
dist = Categorical(probs)
action = dist.sample()
dist.log_prob(action)

tensor([[0.2911, 0.7089]])


/tmp/ipykernel_861795/1647247987.py:2: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probs = F.softmax(torch.randn(1, 2))


tensor([-0.3441])

In [ ]:
def reinforce(env, policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # Help us to calculate the score during the training
    scores_deque = deque(maxlen=100)
    scores = []
    # Line 3 of pseudocode
    for i_episode in range(1, n_training_episodes+1):
        saved_log_probs = []
        rewards = []
        state, info = env.reset()
        # Line 4 of pseudocode
        for t in range(max_t):
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            state, reward, terminated, truncated, info = env.step(action)
            rewards.append(reward)
            done = terminated or truncated
            if done:
                break
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Line 6 of pseudocode: calculate the return
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        # Compute the discounted returns at each timestep,
        # as the sum of the gamma-discounted return at time t (G_t) + the reward at time t

        # In O(N) time, where N is the number of time steps
        # (this definition of the discounted return G_t follows the definition of this quantity
        # shown at page 44 of Sutton&Barto 2017 2nd draft)
        # G_t = r_(t+1) + r_(t+2) + ...

        # Given this formulation, the returns at each timestep t can be computed
        # by re-using the computed future returns G_(t+1) to compute the current return G_t
        # G_t = r_(t+1) + gamma*G_(t+1)
        # G_(t-1) = r_t + gamma* G_t
        # (this follows a dynamic programming approach, with which we memorize solutions in order
        # to avoid computing them multiple times)

        # This is correct since the above is equivalent to (see also page 46 of Sutton&Barto 2017 2nd draft)
        # G_(t-1) = r_t + gamma*r_(t+1) + gamma*gamma*r_(t+2) + ...


        ## Given the above, we calculate the returns at timestep t as:
        #               gamma[t] * return[t] + reward[t]
        #
        ## We compute this starting from the last timestep to the first, in order
        ## to employ the formula presented above and avoid redundant computations that would be needed
        ## if we were to do it from first to last.

        ## Hence, the queue "returns" will hold the returns in chronological order, from t=0 to t=n_steps
        ## thanks to the appendleft() function which allows to append to the position 0 in constant time O(1)
        ## a normal python list would instead require O(N) to do this.
        for t in range(n_steps)[::-1]:
            disc_return_t = (returns[0] if len(returns)>0 else 0)
            returns.appendleft(  gamma * disc_return_t * rewards[t] ) # TODO: complete here

        ## standardization of the returns is employed to make training more stable
        eps = np.finfo(np.float32).eps.item()

        ## eps is the smallest representable float, which is
        # added to the standard deviation of the returns to avoid numerical instabilities
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # Line 7:
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        # Line 8: PyTorch prefers gradient descent
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print('Episode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_deque)))

    return scores
    

In [113]:
from model import Policy

env_id = "CartPole-v1"
env = gym.make(env_id)

observation, info = env.reset()

learning_rate = 1e-4
policy = Policy()
adamw = optim.AdamW(policy.parameters(), lr=learning_rate)


print(observation, info)
print("Observation Shape")

# cart position, velocity, pole angle, angular velocity
print(env.observation_space.sample)

scores = reinforce(
    policy=policy,
    optimizer=adamw,
    n_training_episodes=10000,
    max_t=1000,
    gamma=0.99, #discount factor
    print_every=100
)

[0.00227629 0.0279626  0.00975053 0.02406718] {}
Observation Shape
<bound method Box.sample of Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)>
Episode 100	Average Score: 21.40
Episode 200	Average Score: 20.26
Episode 300	Average Score: 23.08
Episode 400	Average Score: 21.62
Episode 500	Average Score: 21.49
Episode 600	Average Score: 18.80
Episode 700	Average Score: 20.94
Episode 800	Average Score: 22.58
Episode 900	Average Score: 22.12
Episode 1000	Average Score: 20.82
Episode 1100	Average Score: 20.11
Episode 1200	Average Score: 24.16
Episode 1300	Average Score: 21.88
Episode 1400	Average Score: 20.97
Episode 1500	Average Score: 19.79
Episode 1600	Average Score: 22.74
Episode 1700	Average Score: 21.90
Episode 1800	Average Score: 20.58
Episode 1900	Average Score: 21.43
Episode 2000	Average Score: 20.42
Episode 2100	Average Score: 19.85
Episode 2200	Average Score: 23.26
Episode 2300	Average Score: 20.35
Episode 2400	A

In [106]:
def evaluate_agent(env, max_steps, n_eval_episodes, policy):
    """
    Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
    :param env: The evaluation environment
    :param n_eval_episodes: Number of episode to evaluate the agent
    :param policy: The Reinforce agent
    """
    episode_rewards = []
    for episode in range(n_eval_episodes):
        state, info = env.reset()
        step = 0
        done = False
        total_rewards_ep = 0

        for _ in range(max_steps):
            action, _ = policy.act(state)
            new_state, reward, truncated, terminated, info = env.step(action)
            total_rewards_ep += reward

            if done:
                break
            state = new_state
        episode_rewards.append(total_rewards_ep)
    mean_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)

    return mean_reward, std_reward

In [112]:
rewards, std_reward = evaluate_agent(
    env,
    max_steps = 100,
    n_eval_episodes=10,
    policy=policy 
)

print(rewards, std_reward)

20.0 8.06225774829855


In [114]:
import imageio

def record_video(env, policy, out_directory, fps=30):
    """
    Generate a replay video of the agent
    :param env
    :param Qtable: Qtable of our agent
    :param out_directory
    :param fps: how many frame per seconds (with taxi-v3 and frozenlake-v1 we use 1)
    """
    images = []
    done = False
    state = env.reset()
    img = env.render(mode="rgb_array")
    images.append(img)
    while not done:
        # Take the action (index) that have the maximum expected future reward given that state
        action, _ = policy.act(state)
        state, reward, done, info = env.step(action)  # We directly put next_state = state for recording logic
        img = env.render(mode="rgb_array")
        images.append(img)
    imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
record_video(
    env,
    policy,
    out_directory='/eval',
    fps=30
)

In [ ]:
# IN RL 
# 1) we see the environment (observation)
# 2) Agent takes an action based on observation
# 3) Agent gets some form of reward and new state
# 4) Agent updates policy with reward.
# Rinse and Repeat
EPISODES = 100

observation, info = env.reset()
done = False
# Get an initial action
action = policy(torch.from_numpy(observation))


# What happens if we random move. How does our reward look like?
rewards = []
for episode in range(EPISODES):
    episode_reward = 0
    while not done:
        observation, reward, terminated, truncated, info = env.step(np.random.randint(0, 2))
        episode_reward += reward

        # Based on the rewards we get, we must train the policy
        if terminated or truncated:
            break

    rewards.append(episode_reward)
